## The State of Tax Justice: Estimate misalignment for 2017

- Author: Mario Cuenda García, based on Alison Schultz, based on Javier Garcia Bernado's work
- Created: 4 November 2024
- Last updated:

**Description**
- This notebook is the third out of three notebooks to estimate the tax losses caused by profit shifting by multinational enterprises (MNEs). The analysis used the misalignment method based on the country-by-country reports (CbCR) published by the OECD.
    - Details on the misalignment method and its background can be found here: https://www.sciencedirect.com/science/article/pii/S0305750X23003455. 
    - The working paper version is here: https://www.econstor.eu/bitstream/10419/286362/1/wp-2023-33.pdf 

- This notebook estimates profit misalignment based on different formulas. It uses the dataset **"data/final/cbcr_main.csv"** (for the estimation with imputed values) or the dataset **"data/final/cbcr_main_noimputation_allsubgroupsonly.csv"** (for the estimation without imputed values). 

**Outline**
1. Define misalignment
2. Calculate misalignment for sample with full information.
3. Calculate misalignment for samples with imputed data and aggregate results. 

**To dos before running this notebook**
- Run the notebooks 1_clean and 2_impute_missings. Note the requirements and instructions given in these notebooks.

**To dos in this notebook**

Change the formula to any required formula in each section and adapt the output name of the csvs. The formulas I have used are like follows, where sales refer to unrelated party revenues and assets to tangible assets excluding cash.

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]

- Adjust the input path in 5.1 to the bootstrapped sample you use

## 0. Load packages

In [35]:
# Packages
import pandas as pd
import numpy as np
import tjn_tools
from config import *
import os

# Show columns and select data format
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.0f}'.format

## Step 1. Generate the template datasets

### Step 1.1. Generate the dataset with Unique ISO parents

In [36]:
# Open the original dataset
iso_parents = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep just the following columns: iso_parent and year
iso_parents = iso_parents[['iso_parent', 'year']]

# Keep every unique combination of iso_parent and year
iso_parents = iso_parents.drop_duplicates(subset=['iso_parent', 'year'])

# Sort by year, then iso_parent
iso_parents = iso_parents.sort_values(by=['year', 'iso_parent'])

# Filter by year
iso_parents_2016= iso_parents[iso_parents['year'] == 2016]
iso_parents_2017= iso_parents[iso_parents['year'] == 2017]
iso_parents_2018= iso_parents[iso_parents['year'] == 2018]
iso_parents_2019= iso_parents[iso_parents['year'] == 2019]
iso_parents_2020= iso_parents[iso_parents['year'] == 2020]
iso_parents_2021= iso_parents[iso_parents['year'] == 2021]

# OPTIONAL: Print the count of how many unique iso_partner values there are
print(iso_parents_2016['iso_parent'].nunique())
print(iso_parents_2017['iso_parent'].nunique())
print(iso_parents_2018['iso_parent'].nunique())
print(iso_parents_2019['iso_parent'].nunique())
print(iso_parents_2020['iso_parent'].nunique())
print(iso_parents_2021['iso_parent'].nunique())

iso_parents_2017

26
38
46
50
52
52


,iso_parent,year
151,ARG,2017
265,AUS,2017
761,AUT,2017
835,BEL,2017
1064,BMU,2017
1645,BRA,2017
1894,CAN,2017
1985,CHE,2017
2723,CHL,2017
2817,CHN,2017


### Step 1.2. Generate the dataset with unique iso_partners

In [37]:
## Download the relevant dataset
iso_partners = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
iso_partners = iso_partners[~iso_partners['iso_partner'].isin(non_countries)]

# Keep just the following columns: iso_partner and year
iso_partners = iso_partners[['iso_partner', 'year']]

# Sort by year, then iso_partner
iso_partners = iso_partners.sort_values(by=['year', 'iso_partner'])

# Keep every unique combination of iso_partner and year
iso_partners = iso_partners.drop_duplicates(subset=['iso_partner', 'year'])

# Filter by year
iso_partners_2016= iso_partners[iso_partners['year'] == 2016]
iso_partners_2017= iso_partners[iso_partners['year'] == 2017]
iso_partners_2018= iso_partners[iso_partners['year'] == 2018]
iso_partners_2019= iso_partners[iso_partners['year'] == 2019]
iso_partners_2020= iso_partners[iso_partners['year'] == 2020]
iso_partners_2021= iso_partners[iso_partners['year'] == 2021]


# Optional: Print the count of how many unique iso_partner values there are 
print(iso_partners_2016['iso_partner'].nunique())
print(iso_partners_2017['iso_partner'].nunique())
print(iso_partners_2018['iso_partner'].nunique())
print(iso_partners_2019['iso_partner'].nunique())
print(iso_partners_2020['iso_partner'].nunique())
print(iso_partners_2021['iso_partner'].nunique())

183
215
213
210
212
211


### Step 1.3. Generate the template dataset

In [38]:
# Perform a cross join to merge all values of iso_partners_ with each value of iso_parents
iso_combinations_2016 = iso_parents_2016.assign(key=1).merge(iso_partners_2016.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2017 = iso_parents_2017.assign(key=1).merge(iso_partners_2017.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2018 = iso_parents_2018.assign(key=1).merge(iso_partners_2018.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2019 = iso_parents_2019.assign(key=1).merge(iso_partners_2019.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2020 = iso_parents_2020.assign(key=1).merge(iso_partners_2020.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2021 = iso_parents_2021.assign(key=1).merge(iso_partners_2021.assign(key=1), on='key').drop('key', axis=1)

# Concatenate all the years
template_dataset = pd.concat([iso_combinations_2016, iso_combinations_2017, iso_combinations_2018, iso_combinations_2019, iso_combinations_2020, iso_combinations_2021])

# Drop year_y
template_dataset = template_dataset.drop(columns=['year_y'])
# Rename year_x to year
template_dataset = template_dataset.rename(columns={'year_x': 'year'})
# Order columns by iso_parent then iso_partner then year
template_dataset = template_dataset[['iso_parent', 'iso_partner', 'year']]
# Generate new column called cbcr_estimates
template_dataset['cbcr_estimates'] = np.nan

template_dataset

,iso_parent,iso_partner,year,cbcr_estimates
0,AUS,ABW,2016,NaN
1,AUS,AFG,2016,NaN
2,AUS,AGO,2016,NaN
3,AUS,ALB,2016,NaN
4,AUS,AND,2016,NaN
...,...,...,...,...
10967,ZAF,XKV,2021,NaN
10968,ZAF,YEM,2021,NaN
10969,ZAF,ZAF,2021,NaN
10970,ZAF,ZMB,2021,NaN


## Step 2. Define the misalignment formula

In [39]:
def calculate_misalignment(cbcr_data,
                           formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",
                                         'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip'],
                           weights=[.5, 0, 0, .5, 0, 0, 0, 0],
                           profit_var='profit_loss_before_income_tax_corrected',
                           etr_max=0.15): 

    # Create variable with positive profits only for calculating shares
    cbcr_data['profit_var_pos'] = cbcr_data[profit_var]
    cbcr_data.loc[cbcr_data[profit_var] < 0, 'profit_var_pos'] = 0
    cbcr_data['share_profit'] = cbcr_data['profit_var_pos'] / cbcr_data.groupby('iso_parent')['profit_var_pos'].transform('sum')

    # Calculate weighted shares of economic activity
    actual_weights = []
    actual_variables = []
    for i, var in enumerate(formula_vars):
        if var is not None and weights[i] > 0:
            actual_variables.append(f"share_{var}")
            actual_weights.append(weights[i])
            cbcr_data.loc[cbcr_data[var] < 0, var] = 0  # Set economic activity measure to zero if negative
            cbcr_data[f"share_{var}"] = cbcr_data[var] / cbcr_data.groupby('iso_parent')[var].transform('sum')

    # Calculate the share of economic activity
    cbcr_data["share_economy_partner_of_parent"] = (cbcr_data.loc[:, actual_variables] * actual_weights).sum(1, min_count=len(actual_weights))
    # Set economic activity to 1% for those jurisdictions without economic activity but with reported profits
    cbcr_data.loc[(cbcr_data["share_economy_partner_of_parent"] == 0) & (cbcr_data[profit_var] > 0), "share_economy_partner_of_parent"] = 0.01

    # Normalize the economic activity shares to sum to 1
    cbcr_data["share_economy_partner_of_parent"] = cbcr_data["share_economy_partner_of_parent"] / cbcr_data.groupby('iso_parent')["share_economy_partner_of_parent"].transform('sum')

    # Calculate theoretical profit and misaligned profit
    cbcr_data["theoretical_profit"] = cbcr_data["share_economy_partner_of_parent"] * cbcr_data.groupby('iso_parent')[profit_var].transform('sum')
    cbcr_data["misaligned_profit"] = cbcr_data[profit_var] - cbcr_data["theoretical_profit"]

    # Set positive misaligned profits to 0 if ETR exceeds the threshold (etr_max)
    cbcr_data.loc[((cbcr_data["misaligned_profit"] > 0) & (cbcr_data["etr_average_corrected"] > etr_max)), "misaligned_profit"] = 0

    # Adjust misalignment per 'iso_parent'
    def adjust_misalignment(group):
        total_negative_misalignment = group.loc[group["misaligned_profit"] < 0, "misaligned_profit"].sum()
        total_positive_misalignment = group.loc[group["misaligned_profit"] > 0, "misaligned_profit"].sum()
        
        # Adjust negative misalignments to balance positive misalignments within each 'iso_parent'
        if total_negative_misalignment != 0:
            factor = - total_positive_misalignment / total_negative_misalignment
            group.loc[group["misaligned_profit"] < 0, "misaligned_profit"] *= factor
        
        return group

    # Apply the adjustment by grouping by 'iso_parent'
    cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)

    return cbcr_data

## Step 3. Calculate misalignment for sample with full information

### Step 3.1 Import data
- Import data without imputed values. This is only the data from the sample of reporting countries that actually is reported on a country basis, i.e. excluding aggregated country groups and data from reporting countries that do not report on a country by country basis, but just by continents.

In [40]:
cbcr_sample = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
cbcr_sample = cbcr_sample[~cbcr_sample['iso_partner'].isin(non_countries)]

### Step 3.2 Exclude countries that do not report truly country-by-country


- In the 2024 data, the following reporting countries do not report country-by-country. We exclude those from the "clean" analysis where we only use values that are actually in the data.
    - Austria: Only continents in all years
    - Czechia: Only Czechia versus rest of the world from 2019 to 2021
    - Finland: Only Finland and rest of the world between 2016 and 2018 and Finland and continents between 2019 and 2021
    - Greece: Only Greece and continents between 2017 and 2019
    - Hungary: Only Hungary versus rest of the world between 2018 and 2021
    - Isle of Man: Only continents between 2017 and 2020
    - Ireland: Only Ireland versus rest of the world in all years
    - Korea: Only Korea and rest of the world betweem 2016 and 2018 and Korea and continents between 2019 and 2021
    - Macau: Only Macau versus rest of the world between 2019 and 2021
    - Mauritius: Only Mauritius and continents between 2019 and 2021
    - Morocco: Only Morocco and continents in 2021
    - Netherlands: Only Netherlands versus rest of the world between 2016 and 2017
    - Norway: Only Norway and continents 2016 and 2017
    - New Zealand: Only New Zealand versus rest of the world between 2018 and 2021
    - Poland: Only Poland and continents 2019 to 2021
    - Sweden: Only Sweden and continents in all years
    - United Kingdom: Only UK and continents between 2017 and 2021

In [41]:
# Define the conditions for exclusion
exclusion_conditions = [
    ('AUT', 2016, 2021),                # Austria: all years
    ('CZE', 2019, 2021),                # Czechia: from 2019 to 2021
    ('FIN', 2016, 2021),                # Finland: all years
    ('GRC', 2017, 2019),                # Greece: between 2017 and 2019
    ('HUN', 2018, 2021),                # Hungary: between 2018 and 2021
    ('IMN', 2017, 2020),                # Isle of Man: between 2017 and 2020
    ('IRL', 2016, 2021),                # Ireland: all years
    ('KOR', 2016, 2021),                # Korea: all years
    ('MAC', 2019, 2021),                # Macau: between 2019 and 2021
    ('MUS', 2019, 2021),                # Mauritius: between 2019 and 2021
    ('MAR', 2021, 2021),                # Morocco: 2021
    ('NLD', 2016, 2017),                # Netherlands: between 2016 and 2017
    ('NOR', 2016, 2017),                # Norway: 2016 and 2017
    ('NZL', 2018, 2021),                # New Zealand: between 2018 and 2021
    ('POL', 2019, 2021),                # Poland: 2019 to 2021
    ('SWE', 2016, 2021),                # Sweden: all years
    ('GBR', 2017, 2021)                 # United Kingdom: between 2017 and 2021
]

# Iterate through the exclusion conditions
for iso_parent, start_year, end_year in exclusion_conditions:
    cbcr_sample = cbcr_sample[~((cbcr_sample['iso_parent'] == iso_parent) & 
                                     (cbcr_sample['year'].between(start_year, end_year)))]

### Step 3.3 Calculate misalignment for sample countries with full information

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]


The end of this cell shows, among other things, the misaligned profits and the theoretical profits, as well as the CBCR variables.

**More importantly, if for whatever reason we wanted to just run this cell without the "bad reporters", we could just modify the cell to run like the cell from Step 5.2. in order to obtain a dataset of profit shifting without the "bad reporters". In a way, this cell is not really necessary for the rest of the notebook. It just shows an intermediary steps if we want to calculate the misalignment for the "good reporters" only.**

In [42]:
misalignment_2017 = cbcr_sample[cbcr_sample['year'] == 2017].copy()
misalignment_2017 = calculate_misalignment(misalignment_2017, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Keep only the first occurrence of these unique variables for each 'iso_partner'
unique_columns = misalignment_2017.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                               'etr_average_corrected', 'cit',
                                                                               'tax_revenue_current_usd', 
                                                                               'gvt_health_expenditure', 'region_tjn', 
                                                                               'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

# Keep iso_parent, iso_partner, year, misaligned_profit, theoretical_profit, profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
misalignment_2017 = misalignment_2017[['iso_parent', 'iso_partner', 'year', 'misaligned_profit', 'theoretical_profit', 'profit_loss_before_income_tax_corrected', 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']]

# Show if iso_partner = USA
misalignment_2017[misalignment_2017['iso_partner'] == 'USA']

C:\Users\aliso\AppData\Local\Temp\ipykernel_9264\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip
12,ARG,USA,2017,0,"8,273,616","690,931,545",124,"877,803,678","1,496,460,382","5,885,309","1,423,349,011","925,509,422","47,705,744",0
84,AUS,USA,2017,"-4,927,422,530","4,243,937,693","-2,518,141,262","81,992","40,615,480,798","48,510,812,863","3,891,518,407","164,986,000,000","49,678,837,253","9,063,356,454",22
106,BEL,USA,2017,"-12,477,311,760","18,238,511,760","5,761,200,000","48,100","39,537,900,000","15,827,000,000","2,282,930,473","100,908,000,000","49,596,700,000","10,058,900,000",16
196,BMU,USA,2017,"-2,250,758,386","5,214,327,233","1,991,101,178","83,970","56,675,700,057","36,926,954,459","3,985,398,583","52,324,611,878","69,912,708,626","13,237,008,569",22
235,BRA,USA,2017,"-1,330,909,884","8,911,715,675","5,091,349,353","115,961","55,580,421,331","45,643,060,466","5,503,760,927","27,280,709,757","74,000,322,445","18,419,901,114",20
248,CAN,USA,2017,"-32,994,053,291","66,753,823,464","33,301,814,000","620,090","338,193,000,000","355,437,000,000","29,430,818,235","789,903,000,000","413,509,000,000","75,316,117,000",NaN
360,CHE,USA,2017,"-2,102,171,732","16,165,850,826","12,754,393,109","247,164","168,455,000,000","53,748,695,306","11,730,940,280","322,727,000,000","202,530,000,000","34,074,395,450",55
369,CHL,USA,2017,-0,"313,367,478","250,622,258","5,997","4,210,195,169","1,533,639,433","284,630,645","1,985,269,239","4,296,670,407","86,475,238",15
482,CHN,USA,2017,"-2,176,071,887","5,978,994,388","-592,460,171","47,573","65,989,166,649","32,535,940,155","2,257,917,909","47,786,908,233","80,198,909,439","14,209,742,790",38
636,DEU,USA,2017,0,"32,853,002,142","36,944,960,775","610,674","412,661,000,000","210,329,000,000","28,983,914,424","355,390,000,000","522,659,000,000","110,000,000,000",171


## Step 4. Generate the dataset for the "bad reporters"

For the "bad" reporters, we are going to assume that their MNEs behave like the "average" MNE in the countries that report correctly. To do that we need to:

1. First aggregate the variables reported by the CBCR by partner countries. For instance, we see that on aggregate, there are 80m employees reported.
2. Then we look at the share corresponding by partners. For instance, 18m employees are reported in the USA. how many does the USA have. In short, roughly 24% of all employees reported are in the USA.
3. We then assume that 24% of the toal employees reported by the "bad" reporters are assigned to the US. And we repeat with all of them



### Step 4.1. Generate the Total Sums of Variables, and the Total Sums by Partners.

- The first bloc of lines calculates the total of the variables, and generates a new variable (e.g. total_n_employees) in the dataset.
- The second bloc of lines groups by iso_partner and calculates the total sums by partners. (e.g. how many employees are reported by the CBCR countries, say, in the USA)
- The third bloc of lines merges the total sums by partners to the dataset

In [43]:
# Calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_before_income_tax_corrected = misalignment_2017['profit_loss_before_income_tax_corrected'].sum()
total_n_employees = misalignment_2017['n_employees'].sum()
total_unrelated_party_revenues = misalignment_2017['unrelated_party_revenues'].sum()
total_tangible_assets_except_cash = misalignment_2017['tangible_assets_except_cash'].sum()
total_payroll = misalignment_2017['payroll'].sum()
total_stated_capital = misalignment_2017['stated_capital'].sum()
total_total_revenues = misalignment_2017['total_revenues'].sum()
total_related_party_revenues = misalignment_2017['related_party_revenues'].sum()
total_holding_or_managing_ip = misalignment_2017['holding_or_managing_ip'].sum()

misalignment_2017['total_profit_loss_before_income_tax_corrected'] = total_profit_loss_before_income_tax_corrected
misalignment_2017['total_n_employees'] = total_n_employees
misalignment_2017['total_unrelated_party_revenues'] = total_unrelated_party_revenues
misalignment_2017['total_tangible_assets_except_cash'] = total_tangible_assets_except_cash
misalignment_2017['total_payroll'] = total_payroll
misalignment_2017['total_stated_capital'] = total_stated_capital
misalignment_2017['total_total_revenues'] = total_total_revenues
misalignment_2017['total_related_party_revenues'] = total_related_party_revenues
misalignment_2017['total_holding_or_managing_ip'] = total_holding_or_managing_ip

# Group by iso_partner and calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_by_partner = misalignment_2017.groupby('iso_partner')['profit_loss_before_income_tax_corrected'].sum().reset_index()
total_profit_loss_by_partner = total_profit_loss_by_partner.rename(columns={'profit_loss_before_income_tax_corrected': 'total_profit_loss_by_partner'})

total_n_employees_by_partner = misalignment_2017.groupby('iso_partner')['n_employees'].sum().reset_index()
total_n_employees_by_partner = total_n_employees_by_partner.rename(columns={'n_employees': 'total_n_employees_by_partner'})

total_unrelated_party_revenues_by_partner = misalignment_2017.groupby('iso_partner')['unrelated_party_revenues'].sum().reset_index()
total_unrelated_party_revenues_by_partner = total_unrelated_party_revenues_by_partner.rename(columns={'unrelated_party_revenues': 'total_unrelated_party_revenues_by_partner'})

total_tangible_assets_except_cash_by_partner = misalignment_2017.groupby('iso_partner')['tangible_assets_except_cash'].sum().reset_index()
total_tangible_assets_except_cash_by_partner = total_tangible_assets_except_cash_by_partner.rename(columns={'tangible_assets_except_cash': 'total_tangible_assets_except_cash_by_partner'})

total_payroll_by_partner = misalignment_2017.groupby('iso_partner')['payroll'].sum().reset_index()
total_payroll_by_partner = total_payroll_by_partner.rename(columns={'payroll': 'total_payroll_by_partner'})

total_stated_capital_by_partner = misalignment_2017.groupby('iso_partner')['stated_capital'].sum().reset_index()
total_stated_capital_by_partner = total_stated_capital_by_partner.rename(columns={'stated_capital': 'total_stated_capital_by_partner'})

total_total_revenues_by_partner = misalignment_2017.groupby('iso_partner')['total_revenues'].sum().reset_index()
total_total_revenues_by_partner = total_total_revenues_by_partner.rename(columns={'total_revenues': 'total_total_revenues_by_partner'})

total_related_party_revenues_by_partner = misalignment_2017.groupby('iso_partner')['related_party_revenues'].sum().reset_index()
total_related_party_revenues_by_partner = total_related_party_revenues_by_partner.rename(columns={'related_party_revenues': 'total_related_party_revenues_by_partner'})

total_holding_or_managing_ip_by_partner = misalignment_2017.groupby('iso_partner')['holding_or_managing_ip'].sum().reset_index()
total_holding_or_managing_ip_by_partner = total_holding_or_managing_ip_by_partner.rename(columns={'holding_or_managing_ip': 'total_holding_or_managing_ip_by_partner'})

# Merge the total profit loss by partner back into the misalignment_2017 dataframe
misalignment_2017 = misalignment_2017.merge(total_profit_loss_by_partner, on='iso_partner', how='left')
misalignment_2017 = misalignment_2017.merge(total_n_employees_by_partner, on='iso_partner', how='left')
misalignment_2017 = misalignment_2017.merge(total_unrelated_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2017 = misalignment_2017.merge(total_tangible_assets_except_cash_by_partner, on='iso_partner', how='left')
misalignment_2017 = misalignment_2017.merge(total_payroll_by_partner, on='iso_partner', how='left')
misalignment_2017 = misalignment_2017.merge(total_stated_capital_by_partner, on='iso_partner', how='left')
misalignment_2017 = misalignment_2017.merge(total_total_revenues_by_partner, on='iso_partner', how='left')
misalignment_2017 = misalignment_2017.merge(total_related_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2017 = misalignment_2017.merge(total_holding_or_managing_ip_by_partner, on='iso_partner', how='left')

misalignment_2017[misalignment_2017['iso_partner'] == 'USA']

,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner
12,ARG,USA,2017,0,"8,273,616","690,931,545",124,"877,803,678","1,496,460,382","5,885,309","1,423,349,011","925,509,422","47,705,744",0,"4,398,804,655,030","118,500,682","41,911,571,228,709","28,215,190,886,593","3,039,513,854,947","49,116,997,005,594","58,490,415,260,089","16,591,153,472,813","14,583","711,005,776,231","26,957,180","13,207,105,780,809","6,867,171,231,790","1,279,446,313,795","17,494,261,927,876","17,215,353,595,398","4,007,909,358,840","1,648"
84,AUS,USA,2017,"-4,927,422,530","4,243,937,693","-2,518,141,262","81,992","40,615,480,798","48,510,812,863","3,891,518,407","164,986,000,000","49,678,837,253","9,063,356,454",22,"4,398,804,655,030","118,500,682","41,911,571,228,709","28,215,190,886,593","3,039,513,854,947","49,116,997,005,594","58,490,415,260,089","16,591,153,472,813","14,583","711,005,776,231","26,957,180","13,207,105,780,809","6,867,171,231,790","1,279,446,313,795","17,494,261,927,876","17,215,353,595,398","4,007,909,358,840","1,648"
106,BEL,USA,2017,"-12,477,311,760","18,238,511,760","5,761,200,000","48,100","39,537,900,000","15,827,000,000","2,282,930,473","100,908,000,000","49,596,700,000","10,058,900,000",16,"4,398,804,655,030","118,500,682","41,911,571,228,709","28,215,190,886,593","3,039,513,854,947","49,116,997,005,594","58,490,415,260,089","16,591,153,472,813","14,583","711,005,776,231","26,957,180","13,207,105,780,809","6,867,171,231,790","1,279,446,313,795","17,494,261,927,876","17,215,353,595,398","4,007,909,358,840","1,648"
196,BMU,USA,2017,"-2,250,758,386","5,214,327,233","1,991,101,178","83,970","56,675,700,057","36,926,954,459","3,985,398,583","52,324,611,878","69,912,708,626","13,237,008,569",22,"4,398,804,655,030","118,500,682","41,911,571,228,709","28,215,190,886,593","3,039,513,854,947","49,116,997,005,594","58,490,415,260,089","16,591,153,472,813","14,583","711,005,776,231","26,957,180","13,207,105,780,809","6,867,171,231,790","1,279,446,313,795","17,494,261,927,876","17,215,353,595,398","4,007,909,358,840","1,648"
235,BRA,USA,2017,"-1,330,909,884","8,911,715,675","5,091,349,353","115,961","55,580,421,331","45,643,060,466","5,503,760,927","27,280,709,757","74,000,322,445","18,419,901,114",20,"4,398,804,655,030","118,500,682","41,911,571,228,709","28,215,190,886,593","3,039,513,854,947","49,116,997,005,594","58,490,415,260,089","16,591,153,472,813","14,583","711,005,776,231","26,957,180","13,207,105,780,809","6,867,171,231,790","1,279,446,313,795","17,494,261,927,876","17,215,353,595,398","4,007,909,358,840","1,648"
248,CAN,USA,2017,"-32,994,053,291","66,753,823,464","33,301,814,000","620,090","338,193,000,000","355,437,000,000","29,430,818,235","789,903,000,000","413,509,000,000","75,316,117,000",NaN,"4,398,804,655,030","118,500,682","41,911,571,228,709","28,215,190,886,593","3,039,513,854,947","49,116,997,005,594","58,490,415,260,089","16,591,153,472,813","14,583","711,005,776,231","26,957,180","13,207,105,780,809","6,867,171,231,790","1,279,446,313,795","17,494,261,927,876","17,215,353,595,398","4,007,909,358,840","1,648"
360,CHE,USA,2017,"-2,102,171,732","16,165,850,826","12,754,393,109","247,164","168,455,000,000","53,748,695,306","11,730,940,280","322,727,000,000","202,530,000,000

### Step 4.2. Calculate the shares for all the variables

In [44]:
# Final Misalignment
final_misalignment_2017 = misalignment_2017

# Calculate the shares for all variables
final_misalignment_2017['share_reported_total_profit_loss_by_partner'] = misalignment_2017['total_profit_loss_by_partner'] / misalignment_2017['total_profit_loss_before_income_tax_corrected']
final_misalignment_2017['share_reported_total_n_employees_by_partner'] = misalignment_2017['total_n_employees_by_partner'] / misalignment_2017['total_n_employees']
final_misalignment_2017['share_reported_total_unrelated_party_revenues_by_partner'] = misalignment_2017['total_unrelated_party_revenues_by_partner'] / misalignment_2017['total_unrelated_party_revenues']
final_misalignment_2017['share_reported_total_tangible_assets_except_cash_by_partner'] = misalignment_2017['total_tangible_assets_except_cash_by_partner'] / misalignment_2017['total_tangible_assets_except_cash']
final_misalignment_2017['share_reported_total_payroll_by_partner'] = misalignment_2017['total_payroll_by_partner'] / misalignment_2017['total_payroll']
final_misalignment_2017['share_reported_total_stated_capital_by_partner'] = misalignment_2017['total_stated_capital_by_partner'] / misalignment_2017['total_stated_capital']
final_misalignment_2017['share_reported_total_total_revenues_by_partner'] = misalignment_2017['total_total_revenues_by_partner'] / misalignment_2017['total_total_revenues']
final_misalignment_2017['share_reported_total_related_party_revenues_by_partner'] = misalignment_2017['total_related_party_revenues_by_partner'] / misalignment_2017['total_related_party_revenues']
final_misalignment_2017['share_reported_total_holding_or_managing_ip_by_partner'] = misalignment_2017['total_holding_or_managing_ip_by_partner'] / misalignment_2017['total_holding_or_managing_ip']

# Give me column 'share reported' with 2 decimals
pd.options.display.float_format = '{:,.2f}'.format

final_misalignment_2017[final_misalignment_2017['iso_partner'] == 'USA']

,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
12,ARG,USA,2017,0.00,"8,273,615.99","690,931,545.00",124.00,"877,803,678.00","1,496,460,382.00","5,885,309.33","1,423,349,011.00","925,509,422.00","47,705,744.00",0.00,"4,398,804,655,029.77","118,500,682.00","41,911,571,228,708.91","28,215,190,886,592.96","3,039,513,854,946.52","49,116,997,005,593.66","58,490,415,260,088.57","16,591,153,472,813.41","14,583.00","711,005,776,231.00","26,957,180.00","13,207,105,780,809.00","6,867,171,231,790.00","1,279,446,313,794.96","17,494,261,927,876.00","17,215,353,595,398.00","4,007,909,358,840.00","1,648.00",0.16,0.23,0.32,0.24,0.42,0.36,0.29,0.24,0.11
84,AUS,USA,2017,"-4,927,422,530.00","4,243,937,692.93","-2,518,141,262.00","81,992.00","40,615,480,798.00","48,510,812,863.00","3,891,518,406.62","164,986,000,000.00","49,678,837,253.00","9,063,356,454.00",22.00,"4,398,804,655,029.77","118,500,682.00","41,911,571,228,708.91","28,215,190,886,592.96","3,039,513,854,946.52","49,116,997,005,593.66","58,490,415,260,088.57","16,591,153,472,813.41","14,583.00","711,005,776,231.00","26,957,180.00","13,207,105,780,809.00","6,867,171,231,790.00","1,279,446,313,794.96","17,494,261,927,876.00","17,215,353,595,398.00","4,007,909,358,840.00","1,648.00",0.16,0.23,0.32,0.24,0.42,0.36,0.29,0.24,0.11
106,BEL,USA,2017,"-12,477,311,759.87","18,238,511,759.87","5,761,200,000.00","48,100.00","39,537,900,000.00","15,827,000,000.00","2,282,930,473.20","100,908,000,000.00","49,596,700,000.00","10,058,900,000.00",16.00,"4,398,804,655,029.77","118,500,682.00","41,911,571,228,708.91","28,215,190,886,592.96","3,039,513,854,946.52","49,116,997,005,593.66","58,490,415,260,088.57","16,591,153,472,813.41","14,583.00","711,005,776,231.00","26,957,180.00","13,207,105,780,809.00","6,867,171,231,790.00","1,279,446,313,794.96","17,494,261,927,876.00","17,215,353,595,398.00","4,007,909,358,840.00","1,648.00",0.16,0.23,0.32,0.24,0.42,0.36,0.29,0.24,0.11
196,BMU,USA,2017,"-2,250,758,385.97","5,214,327,232.90","1,991,101,178.00","83,970.00","56,675,700,057.00","36,926,954,459.00","3,985,398,582.84","52,324,611,878.00","69,912,708,626.00","13,237,008,569.00",22.00,"4,398,804,655,029.77","118,500,682.00","41,911,571,228,708.91","28,215,190,886,592.96","3,039,513,854,946.52","49,116,997,005,593.66","58,490,415,260,088.57","16,591,153,472,813.41","14,583.00","711,005,776,231.00","26,957,180.00","13,207,105,780,809.00","6,867,171,231,790.00","1,279,446,313,794.96","17,494,261,927,876.00","17,215,353,595,398.00","4,007,909,358,840.00","1,648.00",0.16,0.23,0.32,0.24,0.42,0.36,0.29,0.24,0.11
235,BRA,USA,2017,"-1,330,909,884.45","8,911,715,674.88","5,091,349,353.00","115,961.00","55,580,421,331.00","45,643,060,466.00","5,503,760,927.29","27,280,709,757.00","74,000,322,445.00","18,419,901,114.00",20.00,"4,

### Step 4.3. Keep only the shares for the partners, and drop duplicates (effectively only keep one value for each iso_partner)

In [45]:
# Keep iso_partner and share_reported_total_profit_loss_by_partner	share_reported_total_n_employees_by_partner	share_reported_total_unrelated_party_revenues_by_partner	share_reported_total_tangible_assets_except_cash_by_partner	share_reported_total_payroll_by_partner	share_reported_total_stated_capital_by_partner	share_reported_total_total_revenues_by_partner	share_reported_total_related_party_revenues_by_partner	share_reported_total_holding_or_managing_ip_by_partner
shares_reported_2017 = final_misalignment_2017[['iso_partner', 'share_reported_total_profit_loss_by_partner', 'share_reported_total_n_employees_by_partner', 'share_reported_total_unrelated_party_revenues_by_partner', 'share_reported_total_tangible_assets_except_cash_by_partner', 'share_reported_total_payroll_by_partner', 'share_reported_total_stated_capital_by_partner', 'share_reported_total_total_revenues_by_partner', 'share_reported_total_related_party_revenues_by_partner', 'share_reported_total_holding_or_managing_ip_by_partner']]

# Drop duplicates
shares_reported_2017 = shares_reported_2017.drop_duplicates()

shares_reported_2017[shares_reported_2017['iso_partner'] == 'USA']

,iso_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
12,USA,0.16,0.23,0.32,0.24,0.42,0.36,0.29,0.24,0.11


### Step 4.4. Bring back the excluded countries

- The key here is the third command, where we sum by iso_parent. We basically assume that all countries report for the rest of the world, without caring about continents. **This could be improved and changed**. 
- Once that sum is done, we combine all potential iso_combinations for 2016 (created in Step 1), and we keep all combinations for the "excluded countries".
- Note in the test view, that no matter who is the iso_partner, the number in the variables will be the same because we have aggregated them. The next cells will now create the right shares.


In [46]:
excluded_2017 = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep if year == 2017 and iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_2017 = excluded_2017[excluded_2017['year'] == 2017]
excluded_2017 = excluded_2017[excluded_2017['iso_parent'].isin(['AUT', 'FIN', 'GRC', 'IMN', 'IRL', 'KOR', 'NLD', 'NOR', 'SWE', 'GBR'])]

# 3. Sum by iso_parent: 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected'
excluded_2017 = excluded_2017.groupby('iso_parent').agg({'n_employees': 'sum', 'unrelated_party_revenues': 'sum', 'tangible_assets_except_cash': 'sum', 'payroll': 'sum', 'stated_capital': 'sum', 'total_revenues': 'sum', 'related_party_revenues': 'sum', 'holding_or_managing_ip': 'sum', 'profit_loss_before_income_tax_corrected': 'sum'}).reset_index()

# Merge iso_combinations_2017 with excluded_2017. 
excluded_jurisdictions_2017 = pd.merge(iso_combinations_2017, excluded_2017, on='iso_parent', how='left')

# Drop year_y
excluded_jurisdictions_2017 = excluded_jurisdictions_2017.drop(columns=['year_y'])
# Rename year_x to year
excluded_jurisdictions_2017 = excluded_jurisdictions_2017.rename(columns={'year_x': 'year'})

# Keep if iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_jurisdictions_2017 = excluded_jurisdictions_2017[excluded_jurisdictions_2017['iso_parent'].isin(['AUT', 'FIN', 'GRC', 'IMN', 'IRL', 'KOR', 'NLD', 'NOR', 'SWE', 'GBR'])]

excluded_jurisdictions_2017[excluded_jurisdictions_2017['iso_partner'] == 'USA']


,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected
631,AUT,2017,USA,"1,639,137.00","488,999,046,147.00","283,861,523,108.00","11,665,218,898.12","165,966,590,323.00","622,255,828,387.00","133,347,581,617.00",450.00,"39,214,582,979.65"
2996,FIN,2017,USA,"569,313.00","227,818,000,000.00","88,395,144,252.00","6,605,302,379.33","295,469,136,338.00","317,267,000,000.00","89,460,199,038.00",164.00,"14,524,999,182.00"
3426,GBR,2017,USA,"14,427,427.00","4,274,301,000,000.00","3,060,953,410,797.00","115,560,562,268.16","12,102,856,430,829.00","5,666,038,000,000.00","1,398,623,610,237.00","2,136.00","253,298,825,460.82"
3641,GRC,2017,USA,"519,448.00","481,893,465,848.00","201,101,018,721.00","1,590,285,480.96","473,878,028,683.00","638,607,832,500.00","156,714,788,276.00",79.00,"29,991,537,706.15"
4071,IMN,2017,USA,"74,986.00","14,302,629,515.00","9,128,663,534.00","26,179,850.02","30,132,196,492.00","18,199,759,081.00","3,897,129,617.00",12.00,"387,337,920.67"
4501,IRL,2017,USA,"1,456,465.00","395,125,462,071.00","162,078,883,263.00","4,591,181,770.56","2,794,230,000,000.00","650,537,000,000.00","255,454,136,145.00",411.00,"29,359,192,348.43"
5146,KOR,2017,USA,"3,496,010.00","1,963,138,000,000.00","1,415,425,000,000.00","58,618,739,674.85","527,299,000,000.00","2,798,490,000,000.00","826,447,000,000.00",647.00,"175,448,111,632.23"
6221,NLD,2017,USA,"4,068,007.00","1,786,186,000,000.00","937,366,000,000.00","31,580,472,289.27","2,711,610,000,000.00","2,763,009,000,000.00","976,827,000,000.00",885.00,"112,784,857,351.86"
6436,NOR,2017,USA,"648,470.00","354,275,086,000.00","351,455,432,000.00","12,094,320,926.16","677,316,145,000.00","491,963,281,000.00","137,698,919,000.00",209.00,"67,216,786,660.00"
7726,SWE,2017,USA,"3,236,554.00","856,627,382,193.00","401,828,764,535.00","19,526,751,410.98","318,210,300,113.00","1,365,520,472,885.00","508,893,328,847.00",970.00,"150,110,753,012.63"


### Step 4.5. Merge with the shares reported, and multiply the number

In [47]:
# Merge with share_reported_2017
excluded_jurisdictions_share_reported_2017 = pd.merge(excluded_jurisdictions_2017, shares_reported_2017, on='iso_partner', how='left')

# Multiply 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected' by share_reported
excluded_jurisdictions_share_reported_2017['n_employees'] = excluded_jurisdictions_share_reported_2017['n_employees'] * excluded_jurisdictions_share_reported_2017['share_reported_total_n_employees_by_partner']
excluded_jurisdictions_share_reported_2017['unrelated_party_revenues'] = excluded_jurisdictions_share_reported_2017['unrelated_party_revenues'] * excluded_jurisdictions_share_reported_2017['share_reported_total_unrelated_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2017['tangible_assets_except_cash'] = excluded_jurisdictions_share_reported_2017['tangible_assets_except_cash'] * excluded_jurisdictions_share_reported_2017['share_reported_total_tangible_assets_except_cash_by_partner']
excluded_jurisdictions_share_reported_2017['payroll'] = excluded_jurisdictions_share_reported_2017['payroll'] * excluded_jurisdictions_share_reported_2017['share_reported_total_payroll_by_partner']
excluded_jurisdictions_share_reported_2017['stated_capital'] = excluded_jurisdictions_share_reported_2017['stated_capital'] * excluded_jurisdictions_share_reported_2017['share_reported_total_stated_capital_by_partner']
excluded_jurisdictions_share_reported_2017['total_revenues'] = excluded_jurisdictions_share_reported_2017['total_revenues'] * excluded_jurisdictions_share_reported_2017['share_reported_total_total_revenues_by_partner']
excluded_jurisdictions_share_reported_2017['related_party_revenues'] = excluded_jurisdictions_share_reported_2017['related_party_revenues'] * excluded_jurisdictions_share_reported_2017['share_reported_total_related_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2017['holding_or_managing_ip'] = excluded_jurisdictions_share_reported_2017['holding_or_managing_ip'] * excluded_jurisdictions_share_reported_2017['share_reported_total_holding_or_managing_ip_by_partner']
excluded_jurisdictions_share_reported_2017['profit_loss_before_income_tax_corrected'] = excluded_jurisdictions_share_reported_2017['profit_loss_before_income_tax_corrected'] * excluded_jurisdictions_share_reported_2017['share_reported_total_profit_loss_by_partner']

# Drop share_reported columns
excluded_jurisdictions_dataset_2017 = excluded_jurisdictions_share_reported_2017.drop(columns=[
    'share_reported_total_profit_loss_by_partner',
    'share_reported_total_n_employees_by_partner',
    'share_reported_total_unrelated_party_revenues_by_partner',
    'share_reported_total_tangible_assets_except_cash_by_partner',
    'share_reported_total_payroll_by_partner',
    'share_reported_total_stated_capital_by_partner',
    'share_reported_total_total_revenues_by_partner',
    'share_reported_total_related_party_revenues_by_partner',
    'share_reported_total_holding_or_managing_ip_by_partner'
])

excluded_jurisdictions_dataset_2017[excluded_jurisdictions_dataset_2017['iso_partner'] == 'USA']

,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected
201,AUT,2017,USA,"372,879.80","154,092,579,682.49","69,087,807,810.14","4,910,331,727.73","59,113,202,748.46","183,147,171,461.26","32,212,650,025.64",50.85,"6,338,493,567.60"
416,FIN,2017,USA,"129,510.42","71,789,635,572.32","21,514,105,436.22","2,780,421,535.83","105,238,812,994.04","93,380,489,176.97","21,610,816,243.46",18.53,"2,347,764,706.11"
631,GBR,2017,USA,"3,282,029.61","1,346,910,740,663.17","744,991,989,916.42","48,643,810,316.39","4,310,738,713,368.17","1,667,672,339,497.32","337,864,191,669.91",241.39,"40,942,242,754.34"
846,GRC,2017,USA,"118,166.86","151,853,480,839.57","48,945,092,591.97","669,411,291.94","168,783,656,596.55","187,960,020,396.37","37,857,444,187.30",8.93,"4,847,716,190.97"
1061,IMN,2017,USA,"17,058.22","4,507,021,221.36","2,221,785,273.66","11,020,088.81","10,732,344,606.35","5,356,694,537.68","941,425,940.65",1.36,"62,607,803.83"
1276,IRL,2017,USA,"331,324.58","124,511,289,395.14","39,447,666,645.18","1,932,602,012.24","995,235,753,137.80","191,471,105,686.12","61,709,815,697.12",46.45,"4,745,506,332.35"
1491,KOR,2017,USA,"795,291.38","618,620,835,922.42","344,494,066,328.47","24,674,844,062.39","187,810,887,934.71","823,673,326,116.04","199,644,025,432.76",73.12,"28,358,754,384.94"
1706,NLD,2017,USA,"925,412.37","562,860,011,080.69","228,141,388,613.35","13,293,414,929.71","965,808,548,532.51","813,230,282,444.66","235,971,180,767.08",100.01,"18,230,108,253.81"
1921,NOR,2017,USA,"147,517.48","111,638,585,696.88","85,539,192,046.85","5,090,956,996.84","241,243,291,955.73","144,798,456,306.16","33,263,798,509.64",23.62,"10,864,661,498.51"
2136,SWE,2017,USA,"736,268.92","269,939,019,695.13","97,799,335,932.60","8,219,548,028.22","113,338,654,187.08","401,910,598,137.11","122,932,883,399.53",109.62,"24,263,321,705.89"


## Step 5. Calculate Misalignment with all countries reporting in the CBCR

### Step 5.1. Concatenate the two samples

- Concatenate the cbcr_sample (without the "bad reporters") and the sample with the bad reporters.
- Ensure all required columns are included: 'iso_parent', 'year', 'iso_partner', 'n_employees', 'unrelated_party_revenues', 'tangible_assets_except_cash', 'payroll', 'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip', 'profit_loss_before_income_tax_corrected'.

In [48]:
final_misalignment_2017 = cbcr_sample[cbcr_sample['year'] == 2017].copy()

# Concatenate excluded_jurisdictions_dataset_2016
final_misalignment_2017 = pd.concat([final_misalignment_2017, excluded_jurisdictions_dataset_2017])

# Save the final misalignment dataset
final_misalignment_2017.to_csv(f'{output_tables}/Datasets_to_Perform_Analysis/misalignment_dataset_2017.csv', index=False)

final_misalignment_2017[final_misalignment_2017['iso_partner'] == 'USA']

,iso_parent,parent_jurisdiction,iso_partner,partner_jurisdiction,year,unrelated_party_revenues,profit_loss_before_income_tax,adjusted_profit_loss_before_income_tax,income_tax_paid_on_cash_basis,income_tax_accrued_current_year,n_employees,tangible_assets_except_cash,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,n_cbcr,n_cbcr_groups,n_entities,profit_loss_before_income_tax_corrected,ln_profit_loss_before_income_tax_corrected,ln_unrelated_party_revenues,ln_n_employees,ln_tangible_assets_except_cash,ln_stated_capital,ln_total_revenues,ln_related_party_revenues,ln_holding_or_managing_ip,etr_domestic,etr_domestic_corrected,etr_foreign,etr_foreign_corrected,etr_average,etr_average_corrected,cit,gdp_current_usd,population,gdp,wage_monthly,payroll,ln_wage_monthly,ln_gdp_current_usd,ln_population,gvt_health_expenditure,ln_gvt_health_expenditure,tax_revenue_pct_gdp,tax_revenue_current_usd,cthi_2021_share,cthi_2021_score,region_tjn,ukt,gbr_oct,nld_oct,oecd_oct,oecd,eu
244,ARG,Argentina,USA,United States,2017,"877,803,678.00","690,931,545.00",NaN,"16,999,636.00","15,272,399.00",124.00,"1,496,460,382.00","1,423,349,011.00","925,509,422.00","47,705,744.00",0.00,8.00,8.00,18.00,"690,931,545.00",20.35,20.59,4.83,21.13,21.08,20.65,17.68,0.00,0.16,0.51,0.13,0.13,0.15,0.39,0.27,"19,612,102,000,000.00","325,122,128.00",NaN,"3,955.18","5,885,309.33",8.28,30.61,19.60,"1,671,103,382,743.43",28.14,11.50,"2,256,153,999,452.06",0.01,46.90,Northern America,0.00,0.00,0.00,0.00,1.00,0.00
716,AUS,Australia,USA,United States,2017,"40,615,480,798.00","-2,518,141,262.00",NaN,"318,454,265.00","525,602,966.00","81,992.00","48,510,812,863.00","164,986,000,000.00","49,678,837,253.00","9,063,356,454.00",22.00,81.00,81.00,"1,214.00","-2,518,141,262.00",0.00,24.43,11.31,24.61,25.83,24.63,22.93,3.14,0.16,0.51,0.13,0.13,0.15,0.39,0.27,"19,612,102,000,000.00","325,122,128.00",NaN,"3,955.18","3,891,518,406.62",8.28,30.61,19.60,"1,671,103,382,743.43",28.14,11.50,"2,256,153,999,452.06",0.01,46.90,Northern America,0.00,0.00,0.00,0.00,1.00,0.00
993,BEL,Belgium,USA,United States,2017,"39,537,900,000.00","5,761,200,000.00",NaN,"156,500,000.00","335,900,000.00","48,100.00","15,827,000,000.00","100,908,000,000.00","49,596,700,000.00","10,058,900,000.00",16.00,36.00,36.00,339.00,"5,761,200,000.00",22.47,24.40,10.78,23.48,25.34,24.63,23.03,2.83,0.16,0.51,0.13,0.13,0.15,0.39,0.27,"19,612,102,000,000.00","325,122,128.00",NaN,"3,955.18","2,282,930,473.20",8.28,30.61,19.60,"1,671,103,382,743.43",28.14,11.50,"2,256,153,999,452.06",0.01,46.90,Northern America,0.00,0.00,0.00,0.00,1.00,0.00
1601,BMU,Bermuda,USA,United States,2017,"56,675,700,057.00","1,991,101,178.00",NaN,"405,569,470.00","272,977,421.00","83,970.00","36,926,954,459.00","52,324,611,878.00","69,912,708,626.00","13,237,008,569.00",22.00,46.00,46.00,618.00,"1,991,101,178.00",21.41,24.76,11.34,24.33,24.68,24.97,23.31,3.14,0.16,0.51,0.13,0.13,0.15,0.39,0.27,"19,612,102,000,000.00","325,122,128.00",NaN,"3,955.18","3,985,398,582.84",8.28,30.61,19.60,"1,671,103,382,743.43",28.14,11.50,"2,256,153,999,452.06",0.01,46.90,Northern America,0.00,0.00,0.00,0.00,1.00,0.00
1863,BRA,Brazil,USA,United States,2017,"55,580,421,331.00","5,091,349,353.00",NaN,"866,822,010.00","1,495,377,984.00","115,961.00","45,643,060,466.00","27,280,709,757.00","74,000,322,445.00","18,419,901,114.00",20.00,49.00,49.00,272.00,"5,091,349,353.00",22.35,24.74,11.66,24.54,24.03,25.03,23.64,3.04,0.16,0.51,0.13,0.13,0.15,0.39,0.27,"19,612,102,000,000.00","325,122,128.00",NaN,"3,955.18","5,503,760,927.29",8.28,30.61,19.60,"1,671,103,382,743.43",28.14,11.50,"2,256,153,999,452.06",0.01,46.90,Northern America,0.00,0.00,0.00,0.00,1.00,0.00
1971,CAN,Canada,USA,United States,2017,"338,193,000,000.00","33,301,814,000.00",NaN,"5,013,109,000.00","1,964,382,000.00","620,090.00","355,437,000,000.00","789,903,000,000.00","413,509,000,000.00","75,316,117,000.00",NaN,200.00,200.00,"9,830.00","33,301,814,000.00",24.23,26.55,13.34,26.60,27.40,26.75,25.0

### Step 5.2. Perform the misalignment estimates, and all the remaining calculations needed

In [49]:
# Initialize a list to store the aggregate results
results_sample = []

# Start the estimates
misalignment_final_estimates_2017 = final_misalignment_2017[final_misalignment_2017['year'] == 2017].copy()
misalignment_final_estimates_2017 = calculate_misalignment(misalignment_final_estimates_2017, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Perform the groupby operation on 'iso_partner'
country_results_2017 = misalignment_final_estimates_2017.groupby(['iso_partner']).agg(
    negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
    positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
    theoretical_profit=('theoretical_profit', 'sum'),
    reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
).reset_index()

# Convert results to millions
country_results_2017['negative_misalignment'] = -country_results_2017['negative_misalignment'] / 1e6
country_results_2017['positive_misalignment'] = country_results_2017['positive_misalignment'] / 1e6
country_results_2017['theoretical_profit'] = country_results_2017['theoretical_profit'] / 1e6
country_results_2017['reported_profit'] = country_results_2017['reported_profit'] / 1e6

# Merge the unique columns back into the result
country_results_2017 = country_results_2017.merge(unique_columns, on='iso_partner', how='left')

# Calculate other relevant variables
country_results_2017['tax_revenue_loss'] = country_results_2017['negative_misalignment'] * country_results_2017['cit']
country_results_2017['tax_revenue_gain'] = country_results_2017['positive_misalignment'] * country_results_2017['etr_average_corrected']

country_results_2017['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
    country_results_2017['gvt_health_expenditure'] == 0, 
    np.nan, 
    country_results_2017['tax_revenue_loss'] / (country_results_2017['gvt_health_expenditure'] / 1e6)
)
    
country_results_2017['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
    country_results_2017['tax_revenue_current_usd'] == 0, 
    np.nan, 
    country_results_2017['tax_revenue_loss'] / (country_results_2017['tax_revenue_current_usd'] / 1e6)
)

# Calculate totals
total_positive_misalignment = country_results_2017['positive_misalignment'].sum()
total_negative_misalignment = country_results_2017['negative_misalignment'].sum()
total_profits = country_results_2017['reported_profit'].sum()
misaligned_of_total_profits = total_positive_misalignment / total_profits
total_tax_revenue_loss = country_results_2017['tax_revenue_loss'].sum()
total_tax_revenue_gain = country_results_2017['tax_revenue_gain'].sum()
average_tax_revenue_loss_pct_of_gvt_health_expenditure = country_results_2017['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
average_tax_revenue_loss_pct_of_total_tax_revenues = country_results_2017['tax_revenue_loss_pct_of_total_tax_revenues'].mean()

print(f"Year {2017}: Positive Misalignment: {total_positive_misalignment}, Negative Misalignment: {total_negative_misalignment}, Shifted of total profits: {misaligned_of_total_profits}, "
        f"Total tax revenue loss: {total_tax_revenue_loss}, Total tax revenue gain: {total_tax_revenue_gain}")

# Calculate countries' fractions of totals
country_results_2017['tax_revenue_loss_caused_pct_of_total'] = country_results_2017['positive_misalignment'] / total_positive_misalignment
country_results_2017['tax_revenue_loss_caused_usd'] = country_results_2017['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss
country_results_2017['tax_revenue_loss_suffered_pct_of_total'] = country_results_2017['tax_revenue_loss'] / total_tax_revenue_loss

#country_results_2017 = country_results_2017[['iso_partner', 'partner_jurisdiction', 'negative_misalignment',
#   'tax_revenue_loss', 'tax_revenue_loss_suffered_pct_of_total', 'tax_revenue_loss_pct_of_gvt_health_expenditure',
#   'tax_revenue_loss_pct_of_total_tax_revenues', 'positive_misalignment', 'tax_revenue_gain', 
#   'tax_revenue_loss_caused_usd', 'tax_revenue_loss_caused_pct_of_total', 'etr_average_corrected', 'cit',
#   'region_tjn', 'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

country_results_2017 = country_results_2017.sort_values(by='iso_partner')
country_results_2017.to_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_countries_2017.csv', index=False) # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE
    
# Append aggregate results to the list
results_sample.append({
    'year': 2017,
    'total_positive_misalignment': total_positive_misalignment,
    'total_negative_misalignment': total_negative_misalignment,
    'total_profits': total_profits,
    'misaligned_of_total_profits': misaligned_of_total_profits,
    'total_tax_revenue_loss': total_tax_revenue_loss,
    'total_tax_revenue_gain': total_tax_revenue_gain,
    'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_tax_revenue_loss_pct_of_gvt_health_expenditure,
    'average_tax_revenue_loss_pct_of_total_tax_revenues': average_tax_revenue_loss_pct_of_total_tax_revenues
})

# Convert aggregate results to a DataFrame
results_sample_df = pd.DataFrame(results_sample)

# Save the aggregated results to a CSV or Excel file
results_sample_df.to_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_aggregate_results_2017.csv', index=False)  # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE


Year 2017: Positive Misalignment: 1011907.2580668083, Negative Misalignment: 1011907.2580668081, Shifted of total profits: 0.19197117575543296, Total tax revenue loss: 278862.15401063126, Total tax revenue gain: 71090.88126519261


C:\Users\aliso\AppData\Local\Temp\ipykernel_9264\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


In [50]:
# Ensure the inputs are numeric (optional but robust)
cols_num = ['misaligned_profit', 'profit_loss_before_income_tax_corrected']
misalignment_final_estimates_2017[cols_num] = misalignment_final_estimates_2017[cols_num].apply(
    pd.to_numeric, errors='coerce'
).fillna(0)

# By headquarter (reporting) country
hq_all = (
    misalignment_final_estimates_2017
    .groupby('iso_parent', as_index=False)
    .agg(
        shifted_out=('misaligned_profit', lambda s: (-s.clip(upper=0)).sum()),  # magnitude of negatives
        reported_profit=('profit_loss_before_income_tax_corrected', 'sum'),
        shifted_in=('misaligned_profit', lambda s: s.clip(lower=0).sum())       # optional
    )
)

# Fractions (per HQ)
hq_all['fraction_shifted_out'] = np.where(
    hq_all['reported_profit'] == 0, np.nan, hq_all['shifted_out'] / hq_all['reported_profit']
)
hq_all['fraction_shifted_out_pct'] = 100 * hq_all['fraction_shifted_out']

# (optional)
hq_all['fraction_shifted_in'] = np.where(
    hq_all['reported_profit'] == 0, np.nan, hq_all['shifted_in'] / hq_all['reported_profit']
)
hq_all['fraction_shifted_in_pct'] = 100 * hq_all['fraction_shifted_in']

# Save
hq_all.sort_values('iso_parent').to_csv(
    f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_hq_fractions_2017_all.csv', index=False
)

### Step 5.3 US MNEs

In [51]:
import os
import numpy as np
import pandas as pd

# ================== Config (pick ONE year) ==================
YEAR = 2017  # <— change this per file (e.g., 2018, 2019, …)
OUTPUT_DIR = f"{output_tables}/Final_Full_CBCR_Datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ETR_MAX = 0.15
WEIGHTS = [0.5, 0, 0, 0.5, 0, 0, 0, 0]  # same as your example

# ================== Load the per-year df by name ==================
df_name = f"final_misalignment_{YEAR}"
if df_name not in globals():
    raise NameError(f"{df_name} not found in globals(). Make sure you created it earlier.")
final_misalignment_year = globals()[df_name]

# ================== Collect HQs (keeps your year filter) ==================
hq_list = (
    final_misalignment_year.loc[final_misalignment_year["year"] == YEAR, "iso_parent"]
    .dropna().astype(str).str.upper().unique()
)
hq_list = sorted(hq_list)
print(f"[{YEAR}] HQs: {', '.join(hq_list)}")

# ================== Run for this single year ==================
for HQ in hq_list:
    # 1) Initialize a list to store the aggregate results (per HQ & YEAR)
    results_sample = []

    # 2) Run the estimates — EXACT same filter structure
    mask = (final_misalignment_year['year'] == YEAR) & (final_misalignment_year['iso_parent'] == HQ)
    misalignment_final_estimates = final_misalignment_year.loc[mask].copy()

    if misalignment_final_estimates.empty:
        print(f"[{YEAR}][{HQ}] No rows; skipping.")
        continue

    misalignment_final_estimates = calculate_misalignment(
        misalignment_final_estimates,
        etr_max=ETR_MAX,
        weights=WEIGHTS
    )

    # 3) Groupby iso_partner
    country_results = misalignment_final_estimates.groupby(['iso_partner']).agg(
        negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
        positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
        theoretical_profit=('theoretical_profit', 'sum'),
        reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
    ).reset_index()

    # 4) Convert to millions
    country_results['negative_misalignment'] = -country_results['negative_misalignment'] / 1e6
    country_results['positive_misalignment'] =  country_results['positive_misalignment'] / 1e6
    country_results['theoretical_profit']   =  country_results['theoretical_profit'] / 1e6
    country_results['reported_profit']      =  country_results['reported_profit'] / 1e6

    # 5) Merge the unique columns
    country_results = country_results.merge(unique_columns, on='iso_partner', how='left')

    # 6) Other variables (same logic as yours)
    country_results['tax_revenue_loss'] = country_results['negative_misalignment'] * country_results['cit']
    country_results['tax_revenue_gain'] = country_results['positive_misalignment'] * country_results['etr_average_corrected']

    country_results['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
        country_results['gvt_health_expenditure'] == 0, 
        np.nan, 
        country_results['tax_revenue_loss'] / (country_results['gvt_health_expenditure'] / 1e6)
    )
    country_results['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
        country_results['tax_revenue_current_usd'] == 0, 
        np.nan, 
        country_results['tax_revenue_loss'] / (country_results['tax_revenue_current_usd'] / 1e6)
    )

    # 7) Totals
    total_positive_misalignment = country_results['positive_misalignment'].sum()
    total_negative_misalignment = country_results['negative_misalignment'].sum()
    total_profits               = country_results['reported_profit'].sum()
    misaligned_of_total_profits = total_positive_misalignment / total_profits if total_profits != 0 else np.nan
    total_tax_revenue_loss      = country_results['tax_revenue_loss'].sum()
    total_tax_revenue_gain      = country_results['tax_revenue_gain'].sum()
    average_loss_pct_health     = country_results['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
    average_loss_pct_taxrev     = country_results['tax_revenue_loss_pct_of_total_tax_revenues'].mean()

    print(
        f"[{YEAR}][{HQ}] +mis={total_positive_misalignment:.3f}m, "
        f"-mis={total_negative_misalignment:.3f}m, "
        f"share={misaligned_of_total_profits if pd.notna(misaligned_of_total_profits) else np.nan:.3f}, "
        f"loss={total_tax_revenue_loss:.3f}m, gain={total_tax_revenue_gain:.3f}m"
    )

    # 9) Fractions of totals
    country_results['tax_revenue_loss_caused_pct_of_total'] = (
        country_results['positive_misalignment'] / total_positive_misalignment
        if total_positive_misalignment != 0 else np.nan
    )
    country_results['tax_revenue_loss_caused_usd'] = (
        country_results['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss
    )
    country_results['tax_revenue_loss_suffered_pct_of_total'] = (
        country_results['tax_revenue_loss'] / total_tax_revenue_loss
        if total_tax_revenue_loss != 0 else np.nan
    )

    # Save per-HQ country file
    country_results = country_results.sort_values(by='iso_partner')
    per_hq_file = f'{OUTPUT_DIR}/SOTJ_sample_countries_{YEAR}_{HQ}MNEs.csv'
    country_results.to_csv(per_hq_file, index=False)

    # 10) Append aggregate results to the list (per HQ & year)
    results_sample = [{
        'year': YEAR,
        'total_positive_misalignment': total_positive_misalignment,
        'total_negative_misalignment': total_negative_misalignment,
        'total_profits': total_profits,
        'misaligned_of_total_profits': misaligned_of_total_profits,
        'total_tax_revenue_loss': total_tax_revenue_loss,
        'total_tax_revenue_gain': total_tax_revenue_gain,
        'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_loss_pct_health,
        'average_tax_revenue_loss_pct_of_total_tax_revenues': average_loss_pct_taxrev
    }]

    # 11) Save the aggregated results (per HQ & year)
    results_sample_df = pd.DataFrame(results_sample)
    agg_file = f'{OUTPUT_DIR}/SOTJ_sample_aggregate_results_{YEAR}_{HQ}MNEs.csv'
    results_sample_df.to_csv(agg_file, index=False)


[2017] HQs: ARG, AUS, AUT, BEL, BMU, BRA, CAN, CHE, CHL, CHN, DEU, DNK, ESP, FIN, FRA, GBR, GRC, IDN, IMN, IND, IRL, ITA, JPN, KOR, LUX, LVA, MEX, MYS, NLD, NOR, PER, POL, ROU, SGP, SVN, SWE, USA, ZAF
[2017][ARG] +mis=77.547m, -mis=77.547m, share=0.021, loss=25.124m, gain=7.571m
[2017][AUS] +mis=10210.361m, -mis=10210.361m, share=0.129, loss=2470.936m, gain=895.495m


C:\Users\aliso\AppData\Local\Temp\ipykernel_9264\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
C:\Users\aliso\AppData\Local\Temp\ipykernel_9264\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False)

[2017][AUT] +mis=10174.250m, -mis=10174.250m, share=0.259, loss=2902.735m, gain=1078.230m
[2017][BEL] +mis=121868.130m, -mis=121868.130m, share=0.822, loss=40924.871m, gain=9869.334m


C:\Users\aliso\AppData\Local\Temp\ipykernel_9264\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
C:\Users\aliso\AppData\Local\Temp\ipykernel_9264\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False)

[2017][BMU] +mis=8582.278m, -mis=8582.278m, share=0.376, loss=2249.856m, gain=524.847m
[2017][BRA] +mis=7357.627m, -mis=7357.627m, share=0.124, loss=1142.649m, gain=470.876m


...

[2017][ROU] +mis=60.692m, -mis=60.692m, share=0.085, loss=6.846m, gain=8.587m
[2017][SGP] +mis=96207.279m, -mis=96207.279m, share=0.523, loss=24676.935m, gain=1972.318m
[2017][SVN] +mis=48.039m, -mis=48.039m, share=0.098, loss=7.279m, gain=6.299m


C:\Users\aliso\AppData\Local\Temp\ipykernel_9264\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
C:\Users\aliso\AppData\Local\Temp\ipykernel_9264\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False)

[2017][SWE] +mis=38946.335m, -mis=38946.335m, share=0.259, loss=11111.471m, gain=4127.390m
[2017][USA] +mis=295772.926m, -mis=295772.926m, share=0.272, loss=81636.249m, gain=13586.201m


C:\Users\aliso\AppData\Local\Temp\ipykernel_9264\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
C:\Users\aliso\AppData\Local\Temp\ipykernel_9264\421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False)

[2017][ZAF] +mis=9397.449m, -mis=9397.449m, share=0.236, loss=2319.676m, gain=661.722m


## Step 6. Checking the datasets

### Step 6.1. Checking the countries

In [52]:
# Open the CSV file f'{output_tables}/Scaling_Mario/SOTJ_sample_2016.csv'
sotj_2017_countries = pd.read_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_countries_2017.csv')
sotj_2017_countries

,iso_partner,negative_misalignment,positive_misalignment,theoretical_profit,reported_profit,partner_jurisdiction,etr_average_corrected,cit,tax_revenue_current_usd,gvt_health_expenditure,region_tjn,ukt,oecd,oecd_oct,nld_oct,tax_revenue_loss,tax_revenue_gain,tax_revenue_loss_pct_of_gvt_health_expenditure,tax_revenue_loss_pct_of_total_tax_revenues,tax_revenue_loss_caused_pct_of_total,tax_revenue_loss_caused_usd,tax_revenue_loss_suffered_pct_of_total
0,ABW,"1,047.96",0.00,"6,273.07",65.84,Aruba,0.15,0.25,NaN,NaN,Caribbean/American isl.,0.00,0.00,1.00,1.00,261.99,0.00,NaN,NaN,0.00,0.00,0.00
1,AFG,19.02,0.00,8.40,-20.19,Afghanistan,0.16,0.20,"1,856,301,682.93","121,528,826.51",Asia,0.00,0.00,0.00,0.00,3.80,0.00,0.03,0.00,0.00,0.00,0.00
2,AGO,210.31,139.43,522.15,"1,378.99",Angola,0.42,0.30,"6,797,015,937.22","1,577,680,174.86",Africa,0.00,0.00,0.00,0.00,63.09,57.89,0.04,0.01,0.00,38.42,0.00
3,AIA,0.10,0.00,0.00,-0.40,Anguilla,0.00,0.00,NaN,NaN,Caribbean/American isl.,1.00,0.00,1.00,0.00,0.00,0.00,NaN,NaN,0.00,0.00,0.00
4,ALB,72.95,3.50,66.63,-40.76,Albania,0.07,0.15,"2,459,098,973.29","361,229,769.12",Europe,0.00,0.00,0.00,0.00,10.94,0.24,0.03,0.00,0.00,0.96,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210,XKV,1.66,0.00,3.45,1.02,Kosovo,0.08,0.10,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.17,0.00,NaN,NaN,0.00,0.00,0.00
211,YEM,5.38,0.00,16.88,13.48,Yemen,0.31,0.20,NaN,NaN,Asia,0.00,0.00,0.00,0.00,1.08,0.00,NaN,NaN,0.00,0.00,0.00
212,ZAF,"2,964.58",0.00,"27,113.22","23,627.49",South Africa,0.17,0.28,"94,482,152,765.81","17,400,955,311.21",Africa,0.00,0.00,0.00,0.00,830.08,0.00,0.05,0.01,0.00,0.00,0.00
213,ZMB,817.11,227.43,"1,352.89",-288.10,Zambia,0.14,0.35,"3,928,869,341.86","649,528,657.74",Africa,0.00,0.00,0.00,0.00,285.99,31.51,0.44,0.07,0.00,62.68,0.00


### Step 6.2. Checking the aggregate results

In [53]:
sotj_2017_aggregate = pd.read_csv(f'{output_tables}/Final_Full_CBCR_Datasets/SOTJ_sample_aggregate_results_2017.csv')
sotj_2017_aggregate


,year,total_positive_misalignment,total_negative_misalignment,total_profits,misaligned_of_total_profits,total_tax_revenue_loss,total_tax_revenue_gain,average_tax_revenue_loss_pct_of_gvt_health_expenditure,average_tax_revenue_loss_pct_of_total_tax_revenues
0,2017,"1,011,907.26","1,011,907.26","5,271,141.64",0.19,"278,862.15","71,090.88",0.33,0.38
